# Knowledge Graph Lab: Why a Contract Needs a Graph

**What you will build:** a small web app (Gradio) where you upload a loan agreement, turn it into a **knowledge graph** with Graphify, and then ask the *same question* three ways:

| Mode | What GPT-4o-mini is given |
| --- | --- |
| **A. Plain GPT** | Only your question. It has never seen the contract. |
| **B. Plain RAG** | Your question plus the few contract paragraphs that *sound most similar* to it. |
| **C. Graph RAG** | Your question plus the paragraphs found by **following the links between defined terms** in the graph, plus a calculator for the maths. |

![Plain RAG misses chunks](assets/plain-rag-misses-chunks.png)

**The problem we are solving:** in a contract the answer to a simple question is rarely in one place. "What is the interest rate?" points to *Spread*, which points to *Benchmark*, which points to *Term SOFR*, which points to a *floor*, and a different clause says how many days are in a year. Searching for similar text finds the first page. Following the links finds all of them.

![Graph RAG flow](assets/graph-rag-flow.png)

**The story in one picture (real-world use case):** law firms review loan agreements of 200+ pages and must produce a 7-page abstract of the key obligations. An AI review tool does well on facts written in one place (borrower, lender, closing date, loan amount). It does badly on the **interest rate**, because the rate is never written as one number. It is a *recipe*: Interest Rate = Spread + Benchmark (Term SOFR), and the Benchmark can never be lower than an Index Floor. The tool typically returns only the recipe ("Spread plus Term SOFR") without the Spread value, the Benchmark, or the Floor, so the lawyer still has to dig.

| Step | What a lawyer does | What plain search does | What the knowledge graph does |
| --- | --- | --- | --- |
| 1 | Reads the **Interest Rate** definition: "Spread plus the Benchmark" | Finds this paragraph | Finds this paragraph |
| 2 | Looks up the **Spread** (a number) | Often misses it: the Spread definition does not *sound like* the question | Follows the link Interest Rate to Spread |
| 3 | Looks up the **Benchmark** (Term SOFR) | Often misses it | Follows the link to Benchmark, then Term SOFR |
| 4 | Finds the **Index Floor** that limits Term SOFR | Almost always misses it | Follows the link Term SOFR to Index Floor |
| 5 | Adds it up | Cannot, it has not seen the numbers | Hands the numbers to a calculator |

The app has a tab called **3. The problem and the fix** that runs exactly this story on your contract in one click.

**How to use this notebook:** run the cells from top to bottom. Each code cell has a short plain-English note above it saying what it does and why. The last cell starts the app. You will paste your OpenAI key **inside the app**, not in the notebook.

## Step 1. Install the tools

**What the code below does:** installs the libraries the lab needs.

- `graphifyy` is **Graphify**, the tool that reads documents and turns them into a knowledge graph.
- `gradio` builds the web screen.
- `openai` talks to GPT-4o-mini.
- `pypdf` reads text out of the PDF.
- `numpy` and `networkx` do the maths for search and for storing the graph.

**What to look for:** it may print a few lines and finish. Nothing else is needed.

In [ ]:
%pip install -q "graphifyy[openai,pdf]" gradio openai pypdf numpy networkx

## Step 2. Settings and a place to keep everything

**What the code below does:** sets the model name (`gpt-4o-mini`), imports what we need, and creates one object called `LAB`. Think of `LAB` as a shelf: when you upload a contract, the text pieces, the search index and the graph are all put on this shelf so every later step can find them.

It also defines two tiny helpers: `chat()` (ask GPT a question) and `embed()` (turn text into numbers so we can measure "how similar are these two pieces of text").

**Why we need it:** keeping everything in one place means the app does not need to pass big files around.

**What to look for:** nothing is printed. It just prepares the ground.

In [ ]:
import os, re, sys, json, math, html, time, ast, operator, shutil, subprocess, textwrap
from pathlib import Path
from collections import defaultdict
import numpy as np
import networkx as nx

MODEL = "gpt-4o-mini"                 # the chat model used everywhere in this lab
EMBED_MODEL = "text-embedding-3-small"  # turns text into numbers for similarity search
WORKDIR = Path("lab_work").resolve(); WORKDIR.mkdir(exist_ok=True)


class Lab:
    """The shelf that holds everything built from the uploaded contract."""
    def __init__(self):
        self.reset()

    def reset(self):
        self.client = None        # OpenAI connection
        self.api_key = ""
        self.chunks = []          # the contract cut into clause-sized pieces
        self.chunk_vecs = None    # numbers that let us search the pieces by meaning
        self.graph = None         # the knowledge graph (networkx)
        self.graph_html = ""      # interactive picture of the graph
        self.graphify_html = ""   # Graphify's own picture of its graph
        self.built = False


LAB = Lab()


def make_client(api_key):
    from openai import OpenAI
    return OpenAI(api_key=api_key, base_url=os.environ.get("OPENAI_BASE_URL") or None)


def chat(messages, tools=None, json_mode=False, max_tokens=900, tool_choice=None):
    """Ask GPT-4o-mini. Optionally allow it to call tools or force a JSON reply."""
    kw = dict(model=MODEL, messages=messages, temperature=0, max_tokens=max_tokens)
    if tools:
        kw["tools"] = tools
        if tool_choice:
            kw["tool_choice"] = tool_choice
    if json_mode:
        kw["response_format"] = {"type": "json_object"}
    return LAB.client.chat.completions.create(**kw)


def embed(texts):
    """Turn each text into a list of numbers. Similar meaning gives similar numbers."""
    vecs = []
    for i in range(0, len(texts), 64):
        batch = [(t or " ")[:6000] for t in texts[i:i + 64]]
        r = LAB.client.embeddings.create(model=EMBED_MODEL, input=batch)
        vecs += [d.embedding for d in r.data]
    a = np.array(vecs, dtype="float32")
    a /= (np.linalg.norm(a, axis=1, keepdims=True) + 1e-9)
    return a

## Step 3. Read the whole contract from the PDF

**What the code below does:** `read_pdf()` pulls the text out of every page. `strip_page_furniture()` then removes the clutter that repeats on each page (headers, footers, page numbers) so it does not pollute the clauses.

**Why we need it:** the model can only use text. A real contract is long (the sample has 110 pages), so we read all of it rather than copying a few paragraphs by hand. Nothing about the contract is typed into this notebook.

**What to look for:** later, the app reports the page count and word count.

In [ ]:
def read_pdf(path):
    """Return a list with the text of each page."""
    from pypdf import PdfReader
    return [(p.extract_text() or "") for p in PdfReader(path).pages]


def strip_page_furniture(pages):
    """Join pages into one text, dropping repeated headers/footers and page numbers."""
    def shape(line):                       # "Sch. 6.3.5-3" and "Sch. 6.3.5-4" look the same
        return re.sub(r"\d+", "#", line.strip())

    seen = defaultdict(int)
    for p in pages:
        lines = [l for l in p.split("\n") if l.strip()]
        for s in set(shape(x) for x in (lines[:3] + lines[-3:])):
            seen[s] += 1
    furniture = {s for s, n in seen.items() if n >= max(3, int(0.3 * len(pages)))}

    out = []
    for p in pages:
        lines = p.split("\n")
        n = len(lines)
        keep = []
        for i, l in enumerate(lines):
            s = l.strip()
            at_edge = i < 3 or i >= n - 3
            if at_edge and (shape(l) in furniture or re.fullmatch(r"\d{1,3}", s) or re.fullmatch(r"[ivx]{1,5}", s)):
                continue
            keep.append(re.sub(r"[ \t]+", " ", s))
        out.append("\n".join(keep))
    return "\n".join(out)

## Step 4. Cut the contract into clause-sized pieces

**What the code below does:** `make_chunks()` walks through the contract text and cuts it at natural boundaries:

- every **defined term** ("Interest Rate: shall mean ...") becomes its own piece,
- every **numbered section** ("2.2.3 Computations ...") becomes its own piece,
- very long pieces are split further so none is too big.

Each piece remembers its label (the term or the section number) so we can cite it later.

**Why we need it:** a 110-page contract is far too big to paste into every question. Cutting it into labelled pieces is the first half of **RAG** (retrieval-augmented generation): we hand the model only the pieces it needs. The labels are also what let the graph connect clauses to each other.

**What to look for:** the app log reports how many pieces were made.

In [ ]:
SMALL_WORDS = {"of", "and", "or", "the", "to", "in", "for", "a", "an", "on", "by", "with"}
DEF_RE = re.compile(r"(?m)^([A-Z][^:\n]{1,100}?) ?: (?=(?:shall|means|mean|has|have|is|are)\b)")
SEC_RE = re.compile(r"(?m)^(\d{1,2}(?:\.\d{1,2}){0,3})\.? +([A-Z][A-Za-z;,’'\-/& ]{2,90}?)\.(?: |$)")
ART_RE = re.compile(r"(?m)^(\d{1,2})\. +([A-Z][A-Z ;,&\-]{4,80})$")


def fix_term(term):
    """PDF text sometimes splits a word ('Ra tio'). Glue short lowercase fragments back on."""
    out = []
    for w in term.split():
        if out and w[0].islower() and len(w) <= 4 and w.lower() not in SMALL_WORDS:
            out[-1] += w
        else:
            out.append(w)
    return " ".join(out)


def looks_like_term(term):
    words = re.sub(r'[()"“”]', "", term).split()
    if not 1 <= len(words) <= 10:
        return False
    return all(w[0].isupper() or w.lower() in SMALL_WORDS or not w[0].isalpha() for w in words)


def norm(s):
    return re.sub(r"\s+", " ", s.strip().lower())


def split_long(text, max_chars):
    if len(text) <= max_chars:
        return [text]
    parts, cur = [], ""
    for para in text.split("\n"):
        if len(cur) + len(para) + 1 > max_chars and cur:
            parts.append(cur.strip()); cur = ""
        cur += para + "\n"
    if cur.strip():
        parts.append(cur.strip())
    return parts


def make_chunks(text, max_chars=3500):
    """Cut the contract into labelled pieces (definitions, sections, preamble)."""
    m = re.search(r"(?m)^THIS [A-Z ]*AGREEMENT", text)
    body = text[m.start():] if m else text       # skip cover page and table of contents

    marks = []
    for m in SEC_RE.finditer(body):
        marks.append((m.start(), "section", m.group(1), m.group(2).strip()))
    for m in ART_RE.finditer(body):
        marks.append((m.start(), "article", m.group(1), m.group(2).strip().title()))
    for m in DEF_RE.finditer(body):
        term = fix_term(m.group(1).strip())
        if looks_like_term(term):
            marks.append((m.start(), "definition", term, term))
    marks.sort(key=lambda x: x[0])

    raw = []
    if marks and marks[0][0] > 40:
        raw.append(("section", "preamble", "Parties and loan amount", body[:marks[0][0]].strip()))
    for i, (pos, kind, key, title) in enumerate(marks):
        end = marks[i + 1][0] if i + 1 < len(marks) else len(body)
        raw.append((kind, key, title, body[pos:end].strip()))

    chunks = []
    for kind, key, title, txt in raw:
        if len(txt) < (12 if kind == "definition" else 30):   # short definitions like "Spread: shall mean 4.20%." must survive
            continue
        if kind == "definition":
            label, base, node = key, f"def:{key}", f"term:{norm(key)}"
        elif kind == "article":
            label, base, node = f"Article {key} {title}", f"art:{key}", f"sec:{key}"
        else:
            label, base, node = (f"{key} {title}" if key != "preamble" else title), f"sec:{key}", f"sec:{key}"
        for j, piece in enumerate(split_long(txt, max_chars)):
            chunks.append(dict(id=base if j == 0 else f"{base}#{j+1}", kind=kind, key=key,
                               label=label, node=node, text=piece))
    return chunks

## Step 5. The search index (used by plain RAG, mode B)

**What the code below does:** gives every piece a "meaning fingerprint" (a list of numbers) using `embed()`. `retrieve_plain()` then turns the *question* into a fingerprint too and returns the pieces whose fingerprints are closest.

**Why we need it:** this is how ordinary RAG works. It is good at finding text that *sounds like* the question. It does not know that one clause depends on another.

**What to look for:** in the app, mode B shows exactly which pieces it retrieved ("what the model saw").

In [ ]:
def build_vector_index(chunks):
    return embed([f"{c['label']}\n{c['text']}" for c in chunks])


def retrieve_plain(question, k=4):
    """Plain RAG: the k pieces whose meaning is closest to the question."""
    q = embed([question])[0]
    scores = LAB.chunk_vecs @ q
    top = np.argsort(-scores)[:k]
    return [(LAB.chunks[i], float(scores[i])) for i in top]

## Step 6. First version of the graph: terms and the clauses that use them

**What the code below does:** `build_rule_graph()` creates a dot (node) for every defined term and every section. Then it reads each piece of text and draws a line (edge) whenever it mentions another defined term ("Interest Rate" mentions "Spread") or points to another section ("see Section 2.8").

Terms that appear *everywhere* ("Borrower", "Lender", "Loan") are marked as **common terms**. They would turn the picture into a hairball and they do not help answer questions, so the graph walk ignores them.

**Why we need it:** contracts state their own links on purpose, so we can capture them exactly with simple rules, with no guessing by an AI. Graphify (next step) adds the *meaning-level* links on top.

**What to look for:** the counts of nodes and links, and the list of common terms.

In [ ]:
def build_rule_graph(chunks):
    G = nx.DiGraph()
    for c in chunks:
        n = c["node"]
        if n not in G:
            G.add_node(n, label=c["label"], kind=c["kind"] if c["kind"] != "article" else "section",
                       chunks=[], text="", origin="rules")
        G.nodes[n]["chunks"].append(c["id"])
        G.nodes[n]["text"] += c["text"] + "\n"

    terms = {G.nodes[n]["label"]: n for n in G if G.nodes[n]["kind"] == "definition"}
    term_re = None
    if terms:
        alts = "|".join(re.escape(t) for t in sorted(terms, key=len, reverse=True))
        term_re = re.compile(r"(?<![A-Za-z])(" + alts + r")(?![A-Za-z])")

    def add_edge(a, b, rel):
        if a == b:
            return
        if G.has_edge(a, b):
            G[a][b]["weight"] += 1
        else:
            G.add_edge(a, b, relation=rel, weight=1, origin="rules")

    for c in chunks:
        src, text = c["node"], c["text"]
        if term_re:
            body = text.split(":", 1)[1] if c["kind"] == "definition" and ":" in text[:120] else text
            for m in term_re.finditer(body):
                add_edge(src, terms[m.group(1)], "uses term")
        for m in re.finditer(r"Sections? (\d+(?:\.\d+)*)", text):
            num = m.group(1)
            while num and f"sec:{num}" not in G:
                num = num.rpartition(".")[0]
            if num:
                add_edge(src, f"sec:{num}", "points to section")

    for n in list(G):                                   # section tree: 2.2 contains 2.2.3
        if n.startswith("sec:") and "." in n:
            parent = "sec:" + n[4:].rpartition(".")[0]
            if parent in G:
                add_edge(parent, n, "contains")

    # "common terms": mentioned by a large share of all pieces, e.g. Borrower, Lender, Loan
    limit = max(25, int(0.12 * len(chunks)))
    G.graph["hubs"] = {n for n in G if G.nodes[n]["kind"] == "definition" and G.in_degree(n) >= limit}
    return G

## Step 7. Graphify: let AI find the meaning-level links

**What the code below does:** this is where **Graphify** comes in.

1. `write_graphify_input()` saves the contract pieces as small text files (Graphify reads a folder of documents).
2. `run_graphify()` runs the Graphify tool on that folder using GPT-4o-mini. Graphify reads each file, finds the important concepts (the dots) and the relationships between them (the lines), and saves the result as `graph.json`.
3. `merge_graphify()` joins Graphify's result with the term graph from Step 6, so we have **one** combined graph. Where Graphify and the rules found the same term, they become one dot.

**Why we need it:** the rules in Step 6 catch links the contract states outright. Graphify can also catch links between ideas that are never named explicitly, and it gives us a ready-made interactive picture.

**What to look for:** while it runs you will see Graphify's progress lines. This is the slowest step because it calls GPT for every file, so the app defaults to a **fast demo** that reads only Article 2 (the money terms). Untick it to run Graphify on the whole contract (several minutes).

In [ ]:
def write_graphify_input(chunks, fast=False, target_chars=7000):
    """Save the contract pieces as small markdown files, which is what Graphify reads."""
    src = WORKDIR / "graphify_input"
    shutil.rmtree(src, ignore_errors=True)
    src.mkdir(parents=True)
    use = [c for c in chunks
           if not fast or (c["kind"] != "definition" and (c["key"] == "preamble" or c["key"].split(".")[0] == "2"))]
    files, cur, size = [], [], 0
    for c in use:
        cur.append(c); size += len(c["text"])
        if size >= target_chars:
            files.append(cur); cur, size = [], 0
    if cur:
        files.append(cur)
    for i, group in enumerate(files):
        body = "\n\n".join(f"## {c['label']}\n\n{c['text']}" for c in group)
        (src / f"contract_part_{i:03d}.md").write_text(body, encoding="utf-8")
    return src, len(files)


def _stream(cmd, env, timeout):
    """Run a command and yield its output lines as they appear. Returns the exit code."""
    proc = subprocess.Popen(cmd, cwd=WORKDIR, env=env, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    start = time.time()
    for line in proc.stdout:
        yield ("log", line.rstrip())
        if time.time() - start > timeout:
            proc.kill()
            yield ("log", "Graphify took too long and was stopped.")
            break
    return proc.wait()


def run_graphify(src, api_key, timeout=900):
    """Run `graphify extract` on the folder. Yields progress lines, then the path of graph.json."""
    env = dict(os.environ, OPENAI_API_KEY=api_key, GRAPHIFY_BACKEND="openai", GRAPHIFY_OPENAI_MODEL=MODEL)
    cmd = [sys.executable, "-m", "graphify", "extract", str(src), "--backend", "openai", "--model", MODEL]
    code = yield from _stream(cmd, env, timeout)
    gp = Path(src) / "graphify-out" / "graph.json"
    yield ("done", gp if gp.exists() else None, code)


def merge_graphify(G, graph_json_path):
    """Add Graphify's concepts and relationships into our graph G. Returns a few counts."""
    d = json.loads(Path(graph_json_path).read_text(encoding="utf-8"))
    g_nodes = d.get("nodes") or []
    g_edges = d.get("links") or d.get("edges") or []
    by_label = {norm(G.nodes[n]["label"]): n for n in G if G.nodes[n]["kind"] == "definition"}
    idmap, new_nodes, added = {}, 0, 0
    for n in g_nodes:
        gid = n.get("id")
        label = str(n.get("label") or n.get("name") or gid)
        key = norm(label)
        if key in by_label:
            nid = by_label[key]                      # same term found by the rules: one dot
        else:
            nid = f"concept:{key}"
            if nid not in G:
                G.add_node(nid, label=label, kind="concept", chunks=[], origin="graphify",
                           text=f"(Concept found by Graphify in {n.get('source_file') or 'the contract'}.)")
                new_nodes += 1
        idmap[gid] = nid
    for e in g_edges:
        a = idmap.get(e.get("source", e.get("from")))
        b = idmap.get(e.get("target", e.get("to")))
        if not a or not b or a == b:
            continue
        rel = str(e.get("relation") or e.get("label") or "related to").replace("_", " ")
        if G.has_edge(a, b):
            G[a][b]["weight"] += 1
            G[a][b]["graphify_relation"] = rel
        else:
            G.add_edge(a, b, relation=rel, weight=1, origin="graphify")
            added += 1
    return dict(graphify_nodes=len(g_nodes), graphify_edges=len(g_edges),
                new_concept_nodes=new_nodes, new_edges=added)

## Step 8. Draw the graph so you can play with it

**What the code below does:** `render_graph_html()` turns the graph into an interactive picture inside the page. Blue dots are **defined terms**, orange dots are **sections**, green dots are **concepts Graphify found**. Bigger dots have more links.

In the picture you can **drag** dots around, **zoom** with the mouse wheel, **type a term** in the search box, and **click** a dot to read the contract text behind it and see what it links to.

**Why we need it:** seeing the web of clauses is the whole point of the lesson. Try searching "Interest Rate" and look at where the lines go.

**What to look for:** the "common terms" (Borrower, Lender, Loan, ...) are left out of the picture on purpose, because they link to almost everything.

![Terms vs knowledge graph](assets/terms-vs-knowledge-graph.png)

In [ ]:
KIND_COLOR = {"definition": "#4C78A8", "section": "#F58518", "concept": "#54A24B"}

GRAPH_TEMPLATE = """<!doctype html><html><head><meta charset="utf-8">
<script src="https://cdnjs.cloudflare.com/ajax/libs/vis-network/9.1.9/standalone/umd/vis-network.min.js"></script>
<style>
body{margin:0;font-family:Arial,Helvetica,sans-serif;font-size:13px;background:#ffffff;color:#111111}
#bar{background:#f4f4f4;color:#111;padding:6px 8px;border-bottom:1px solid #ddd;display:flex;gap:10px;align-items:center;flex-wrap:wrap}
#q{padding:4px 6px;width:220px;background:#fff;color:#111;border:1px solid #bbb}
.dot{display:inline-block;width:10px;height:10px;border-radius:50%;margin-right:3px}
#wrap{display:flex;height:__H__px}
#net{flex:1;min-width:0;background:#ffffff}
#side{width:330px;overflow:auto;padding:8px;border-left:1px solid #ddd;font-size:12px;background:#ffffff;color:#111111}
#side pre{white-space:pre-wrap;font-family:inherit;background:#f3f3f3;color:#111;padding:6px;border-radius:4px}
.tag{display:inline-block;color:#111;background:#e6e9ff;border-radius:10px;padding:1px 7px;margin:1px;cursor:pointer}
</style></head><body>
<div id="bar">
 <input id="q" list="names" placeholder="Search a term, e.g. Interest Rate">
 <datalist id="names"></datalist>
 <label><input type="checkbox" id="freeze"> freeze layout</label>
 <span><span class="dot" style="background:#4C78A8"></span>term
 <span class="dot" style="background:#F58518;margin-left:8px"></span>section
 <span class="dot" style="background:#54A24B;margin-left:8px"></span>Graphify concept
 <span class="dot" style="background:#fff;border:3px solid #d62728;margin-left:8px"></span>used for the answer</span>
</div>
<div id="wrap"><div id="net"></div><div id="side">Click a dot to read its clause and see its links.</div></div>
<script>
const NODES=__NODES__, EDGES=__EDGES__, TEXT=__TEXT__, SHOW_ALL=__SHOWALL__;
const nodes=new vis.DataSet(NODES), edges=new vis.DataSet(EDGES);
const net=new vis.Network(document.getElementById('net'),{nodes,edges},{
  nodes:{shape:'dot',font:{size:13,face:'Arial',strokeWidth:3,strokeColor:'#ffffff'}},
  edges:{smooth:{type:'continuous'},arrows:{to:{scaleFactor:0.5}}},
  physics:{solver:'forceAtlas2Based',forceAtlas2Based:{gravitationalConstant:-45,springLength:90},stabilization:{iterations:200}},
  interaction:{hover:true,navigationButtons:true,keyboard:true,tooltipDelay:120}});
const names=NODES.map(n=>n.name);
document.getElementById('names').innerHTML=names.map(n=>'<option value="'+n.replace(/"/g,'&quot;')+'">').join('');
function baseLabel(n){return SHOW_ALL?n.name:(n.deg>=4?n.name:'');}
function resetLabels(){nodes.update(NODES.map(n=>({id:n.id,label:baseLabel(n)})));}
function esc(s){return s.replace(/[&<>]/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;'}[c]));}
function tags(ids){return ids.map(i=>'<span class="tag" data-id="'+i.replace(/"/g,'&quot;')+'">'+esc(nodes.get(i).name)+'</span>').join(' ')||'none';}
function focusNode(id){
  resetLabels();
  const nb=net.getConnectedNodes(id);
  nodes.update([id].concat(nb).map(i=>({id:i,label:nodes.get(i).name})));
  const n=nodes.get(id), out=net.getConnectedNodes(id,'to'), inn=net.getConnectedNodes(id,'from');
  document.getElementById('side').innerHTML='<b>'+esc(n.name)+'</b> <i>('+n.kind+')</i>'
   +'<p><b>Links to:</b> '+tags(out)+'</p><p><b>Linked from:</b> '+tags(inn)+'</p>'
   +'<pre>'+esc(TEXT[id]||'')+'</pre>';
  document.querySelectorAll('.tag').forEach(t=>t.onclick=()=>jump(t.dataset.id));
}
function jump(id){net.selectNodes([id]);net.focus(id,{scale:0.8,animation:true});focusNode(id);}
net.on('click',p=>{if(p.nodes.length)focusNode(p.nodes[0]);});
document.getElementById('q').addEventListener('change',e=>{
  const q=e.target.value.trim().toLowerCase(); if(!q)return;
  const hit=NODES.find(n=>n.name.toLowerCase()===q)||NODES.find(n=>n.name.toLowerCase().includes(q));
  if(hit)jump(hit.id);
});
document.getElementById('freeze').onchange=e=>net.setOptions({physics:!e.target.checked});
resetLabels();
</script></body></html>"""


def graph_payload(G, subset=None, highlight=()):
    hubs = G.graph.get("hubs", set())
    keep = [n for n in G if (n in subset if subset is not None else n not in hubs)]
    keepset = set(keep)
    nodes, text = [], {}
    for n in keep:
        d = G.nodes[n]
        deg = sum(1 for m in G.predecessors(n) if m in keepset) + sum(1 for m in G.successors(n) if m in keepset)
        col = KIND_COLOR.get(d["kind"], "#999999")
        hl = n in highlight
        nodes.append({"id": n, "name": d["label"], "label": d["label"], "kind": d["kind"], "deg": deg,
                      "title": d["label"], "color": {"background": col, "border": "#d62728" if hl else col},
                      "borderWidth": 4 if hl else 1, "size": 8 + 3 * math.sqrt(max(deg, 1)) + (8 if hl else 0)})
        text[n] = d.get("text", "")[:4000]
    edges = []
    for a, b, d in G.edges(data=True):
        if a in keepset and b in keepset:
            edges.append({"from": a, "to": b, "title": d.get("graphify_relation") or d.get("relation", ""),
                          "color": {"color": "#9a9a9a", "opacity": 0.6}, "width": min(1 + 0.3 * d.get("weight", 1), 4)})
    return nodes, edges, text


def render_graph_html(G, subset=None, highlight=(), height=620):
    nodes, edges, text = graph_payload(G, subset, highlight)
    dump = lambda o: json.dumps(o).replace("</", "<\\/")
    return (GRAPH_TEMPLATE.replace("__H__", str(height)).replace("__NODES__", dump(nodes))
            .replace("__EDGES__", dump(edges)).replace("__TEXT__", dump(text))
            .replace("__SHOWALL__", "true" if subset is not None else "false"))


def as_iframe(doc, height=650):
    """Put a full web page inside a box so it can be shown in Gradio or in the notebook."""
    return (f'<iframe srcdoc="{html.escape(doc, quote=True)}" '
            f'style="width:100%;height:{height}px;border:1px solid #ccc;border-radius:6px;background:#ffffff;color-scheme:light"></iframe>')


def graph_stats(G):
    kinds = defaultdict(int)
    for n in G:
        kinds[G.nodes[n]["kind"]] += 1
    hubs = ", ".join(sorted(G.nodes[h]["label"] for h in G.graph.get("hubs", [])))
    return (f"**{G.number_of_nodes()} nodes** ({kinds['definition']} defined terms, {kinds['section']} sections, "
            f"{kinds['concept']} Graphify concepts) and **{G.number_of_edges()} links**. "
            f"Left out of the picture as too common: {hubs or 'none'}.")

## Step 9. A calculator the model can call

**What the code below does:** defines two small Python functions the model is allowed to call:

- `interest_calculator()` applies the loan's interest rules: *the market rate can never count as lower than the floor; the annual rate is the Spread plus that market rate; the daily rate is the annual rate divided by the day-count year; interest is principal x daily rate x number of days.*
- `calculate()` evaluates a plain arithmetic expression safely.

**Why we need it:** language models are unreliable at arithmetic. So the model finds the *numbers and rules* (from the contract) and a calculator does the sum. The same calculator is available in all three modes, so the only difference between the modes is the **context**.

**What to look for:** in the app, every answer shows which calculator calls were made and what numbers went in. If a mode did not read the contract, you can see it feed in guessed numbers.

In [ ]:
def interest_calculator(principal, spread_pct, benchmark_pct, index_floor_pct=0.0, days=30, day_count_basis=360):
    """Interest for a period on a floating-rate loan with a floor on the market rate."""
    benchmark_used = max(benchmark_pct, index_floor_pct)
    annual = spread_pct + benchmark_used
    daily = annual / 100 / day_count_basis
    interest = principal * daily * days
    return {"benchmark_used_pct": round(benchmark_used, 5), "annual_rate_pct": round(annual, 5),
            "daily_rate_pct": round(daily * 100, 8), "daily_interest": round(principal * daily, 2), "days": days, "day_count_basis": day_count_basis,
            "interest_for_period": round(interest, 2)}


_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
        ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos}


def calculate(expression):
    """Safely evaluate plain arithmetic such as '13817798 * 0.082 * 30 / 360'."""
    def ev(n):
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)):
            return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS:
            return _OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in _OPS:
            return _OPS[type(n.op)](ev(n.operand))
        raise ValueError("only plain arithmetic is allowed")
    return {"result": round(ev(ast.parse(expression, mode="eval").body), 6)}


TOOLS = [
    {"type": "function", "function": {
        "name": "interest_calculator",
        "description": "Compute interest for a period on a floating-rate loan. The market rate (benchmark) is never "
                       "allowed to count as lower than the index floor. Rates are in percent (4.2 means 4.2%).",
        "parameters": {"type": "object", "properties": {
            "principal": {"type": "number"}, "spread_pct": {"type": "number"},
            "benchmark_pct": {"type": "number", "description": "market rate, e.g. Term SOFR, in percent"},
            "index_floor_pct": {"type": "number", "description": "minimum the benchmark can count as, in percent"},
            "days": {"type": "number"}, "day_count_basis": {"type": "number", "description": "days in a year, e.g. 360"}},
            "required": ["principal", "spread_pct", "benchmark_pct", "days", "day_count_basis"]}}},
    {"type": "function", "function": {
        "name": "calculate", "description": "Evaluate a plain arithmetic expression.",
        "parameters": {"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]}}},
]


def run_tool(name, args):
    try:
        if name == "interest_calculator":
            return interest_calculator(**args)
        if name == "calculate":
            return calculate(**args)
        return {"error": f"unknown tool {name}"}
    except Exception as e:
        return {"error": str(e)}

## Step 10. The three ways of answering

**What the code below does:** three functions, one per mode. They all ask GPT-4o-mini the same question and offer the same calculator. They differ only in **what text they hand over**:

- `ask_plain()` hands over **nothing**. This is mode A.
- `ask_rag()` hands over the **4 most similar pieces** (`retrieve_plain`). This is mode B.
- `ask_graph()` is mode C. It does a **graph walk**: it finds the terms in your question (the *starting dots*), then repeats up to four times: look at the dots linked to what it has opened so far, let GPT-4o-mini pick which linked ones it needs, and open them. The result is the chain of clauses a person would follow by hand.

Two safety checks run around the answer, in both B and C. The **number check** compares every number in the answer with the contract text, the question and the calculator results, and asks the model to correct any it cannot trace. Mode C also runs an **evidence check** before answering: the model lists the facts the question needs and quotes where each one appears, Python verifies each quote really is in the opened clauses, and anything missing sends the walk back to open more clauses.

`llm_with_tools()` is the small loop that lets the model call the calculator and read the result.

**Why we need it:** this is the heart of the lab. Mode B can only find text that resembles the question. Mode C follows the links the contract itself creates.

**What to look for:** in the app, mode C prints the **trail** it followed ("Round 1 opened: Spread, Benchmark ...") and highlights those dots in a mini-graph.

In [ ]:
SYSTEM_PLAIN = ("You are a helpful finance assistant. You have NOT been given the loan agreement. "
                "Answer from general knowledge only. If the answer depends on terms of a specific contract you "
                "cannot see, say so clearly instead of guessing. Use the calculator tools for any arithmetic. "
                "Write plain text only, no LaTeX.")

SYSTEM_RAG = ("You answer questions about ONE loan agreement using ONLY the contract excerpts provided.\n"
              "- Quote numbers exactly as written and cite the clause in brackets, e.g. [Spread] or [2.2.3].\n"
              "- If the excerpts do not contain what is needed, say what is missing instead of guessing.\n"
              "- ALWAYS do arithmetic with the interest_calculator or calculate tool, even simple additions. "
              "Never state a computed number that did not come from a tool.\n"
              "- If a rule in the contract sets a floor on a rate, apply it through the calculator's index_floor_pct input.\n"
              "- For questions about the lowest or minimum possible rate, use the contract's floor, not the user's "
              "assumed market rate. Use the assumed market rate only when the question needs a current or specific rate.\n"
              "- For any question about an interest rate, always state all four parts: the formula from the contract, the "
              "Spread value, the Benchmark (and its definition), and the floor on the Benchmark with its value. Then give the "
              "resulting rate for the assumed market rate by calling interest_calculator with the Spread, the market rate and "
              "index_floor_pct, so the floor is applied and visible. A rate above the floor must still mention the floor.\n"
              "- Only call a calculator with numbers that appear in the excerpts or in the user's assumptions. If a value is "
              "missing from the excerpts, do not invent one and do not call a calculator: say what is missing.\n"
              "- Write plain text only, no LaTeX. Give a short answer first, then the steps.")


def assumptions_text(principal, sofr_pct, days):
    return (f"User assumptions for any calculation: principal = ${principal:,.2f}; days in the period = {days}. "
            f"If the question needs a current Term SOFR value and does not state one, assume {sofr_pct}% (Term SOFR is a "
            f"market rate that is not stated in the contract).")


NEEDS_MATH = re.compile(r"how much|calculate|compute|interest (for|on|accru|would|is)|amount of interest|total|\$", re.I)


def llm_with_tools(system, user, max_rounds=4):
    """Ask the model; if it calls a calculator, run it and give the result back.
    If the question needs arithmetic and the model skipped the calculator, make it redo the answer with the tool."""
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    question = user.split("\n\n")[0]
    calls, forced = [], False
    for _ in range(max_rounds + 1):
        r = chat(msgs, tools=TOOLS, tool_choice="required" if forced else None)
        m = r.choices[0].message
        if not m.tool_calls:
            if not calls and not forced and NEEDS_MATH.search(question):
                forced = True                      # the model did the sums itself: send it back to the calculator
                msgs.append({"role": "assistant", "content": m.content or ""})
                msgs.append({"role": "user", "content": "You did arithmetic without a calculator. Call the calculator "
                             "tool for every number you compute, then answer again."})
                continue
            return m.content or "", calls
        forced = False
        msgs.append({"role": "assistant", "content": m.content or "", "tool_calls": [
            {"id": t.id, "type": "function", "function": {"name": t.function.name, "arguments": t.function.arguments}}
            for t in m.tool_calls]})
        for t in m.tool_calls:
            try:
                args = json.loads(t.function.arguments or "{}")
            except Exception:
                args = {}
            out = run_tool(t.function.name, args)
            calls.append({"tool": t.function.name, "input": args, "output": out})
            msgs.append({"role": "tool", "tool_call_id": t.id, "content": json.dumps(out)})
    msgs.append({"role": "user", "content": "Give your final answer now."})
    return chat(msgs).choices[0].message.content or "", calls


def ask_plain(question, assumptions):
    answer, calls = llm_with_tools(SYSTEM_PLAIN, f"Question: {question}\n\n{assumptions}")
    return dict(answer=answer, calls=calls, saw="(nothing: the model was given only the question)")


def find_seeds(question, max_seeds=3):
    """Pick the starting dots: terms named in the question (or the closest-meaning clauses if none)."""
    G = LAB.graph
    hubs = G.graph.get("hubs", set())
    q = " " + re.sub(r"[^a-z0-9§. /&-]", " ", question.lower()) + " "
    seeds = []
    for m in re.finditer(r"section (\d+(?:\.\d+)*)", q):
        if f"sec:{m.group(1)}" in G:
            seeds.append(f"sec:{m.group(1)}")
    terms = [(n, norm(d["label"])) for n, d in G.nodes(data=True) if d["kind"] == "definition" and n not in hubs]
    for n, lab in sorted(terms, key=lambda x: -len(x[1])):
        if f" {lab} " in q and not any(lab in norm(G.nodes[s]["label"]) for s in seeds):
            seeds.append(n)
        if len(seeds) >= max_seeds:
            break
    if not seeds:                              # no term named: fall back to the closest-meaning clauses
        qv = embed([question])[0]
        for i in np.argsort(-(LAB.chunk_vecs @ qv))[:max_seeds]:
            n = LAB.chunks[i]["node"]
            if n not in seeds and n not in hubs:
                seeds.append(n)
    return seeds


def snippet(G, n, width=140):
    t = G.nodes[n]["text"].strip().replace("\n", " ")
    t = t.split(":", 1)[1].strip() if G.nodes[n]["kind"] == "definition" and ":" in t[:120] else t
    return t[:width]


def candidate_nodes(opened, limit=24):
    """Dots linked to what we have already opened (ignoring very common terms)."""
    G = LAB.graph
    hubs = G.graph.get("hubs", set())
    done, score = set(opened), defaultdict(float)
    for n in opened:
        for m in G.successors(n):
            if m not in done and m not in hubs and G.nodes[m]["kind"] != "concept":
                score[m] += G[n][m]["weight"]
        for m in G.predecessors(n):
            if m not in done and G.nodes[m]["kind"] == "section":
                score[m] += 0.5 * G[m][n]["weight"]
    return sorted(score, key=lambda m: -score[m])[:limit]


def choose_nodes(question, opened, cands, per_round, qvec):
    """Let GPT-4o-mini decide which linked dots are needed. Falls back to closest meaning."""
    G = LAB.graph
    listing = "\n".join(f"{i+1}. {G.nodes[m]['label']}: {snippet(G, m)}" for i, m in enumerate(cands))
    prompt = (f"Question about a loan agreement: {question}\n\nAlready opened: "
              f"{', '.join(G.nodes[o]['label'] for o in opened)}\n\nLinked clauses you could open next:\n{listing}\n\n"
              f"Return JSON like {{\"open\": [1, 4]}} listing up to {per_round} numbers of clauses that are needed "
              f"to answer or to compute the answer (rates, floors, day counts, dates, amounts). "
              f"Return {{\"open\": []}} if none are needed.")
    try:
        r = chat([{"role": "system", "content": "You follow links between defined terms in a contract."},
                  {"role": "user", "content": prompt}], json_mode=True, max_tokens=120)
        nums = json.loads(r.choices[0].message.content).get("open", [])
        picks = [cands[int(i) - 1] for i in nums if 1 <= int(i) <= len(cands)]
        return picks[:per_round]
    except Exception:
        idx = {n: i for i, c in enumerate(LAB.chunks) for n in [c["node"]]}
        ranked = sorted(cands, key=lambda m: -float(LAB.chunk_vecs[idx[m]] @ qvec) if m in idx else 0)
        return ranked[:per_round]


def graph_retrieve(question, rounds=4, per_round=6):
    """The graph walk. Returns the opened dots (in order) and a plain-English trail."""
    G = LAB.graph
    seeds = find_seeds(question)
    opened, trail = list(seeds), [f"Start: {', '.join(G.nodes[s]['label'] for s in seeds) or 'nothing matched'}"]
    qvec = embed([question])[0]
    for r in range(rounds):
        cands = candidate_nodes(opened)
        if not cands:
            break
        picks = [p for p in choose_nodes(question, opened, cands, per_round, qvec) if p not in opened]
        if not picks:
            trail.append(f"Round {r+1}: the model saw {len(cands)} linked clauses and needed no more.")
            break
        opened += picks
        trail.append(f"Round {r+1}: opened {', '.join(G.nodes[p]['label'] for p in picks)}")
    return opened, trail


CLAUSE_BUDGET = 5200      # most characters of any one clause sent to the model


def clause_context(node, qvec, budget=CLAUSE_BUDGET):
    """The text of one dot. If a long clause was split into pieces, keep the pieces closest to the question."""
    idx = [i for i, c in enumerate(LAB.chunks) if c["node"] == node]
    if sum(len(LAB.chunks[i]["text"]) for i in idx) > budget:
        keep, used = [], 0
        for i in sorted(idx, key=lambda i: -float(LAB.chunk_vecs[i] @ qvec)):
            n = len(LAB.chunks[i]["text"])
            if not keep or used + n <= budget:
                keep.append(i); used += n
        idx = sorted(keep)
    return "\n".join(LAB.chunks[i]["text"] for i in idx)


def squash(s):
    """Lower-case and keep only letters, digits and a few symbols, so small spacing differences do not matter."""
    return re.sub(r"[^a-z0-9.%$]", "", s.lower())


def build_context(nodes, qvec):
    parts = [(LAB.graph.nodes[n]["label"], clause_context(n, qvec)) for n in nodes]
    return parts, "\n\n".join(f"[{label}]\n{text}" for label, text in parts)


def complete_evidence(question, opened, qvec, trail, max_loops=2):
    """Check that every fact the question needs is actually in the opened clauses; if not, open more."""
    G = LAB.graph
    for _ in range(max_loops):
        parts, ctx = build_context(opened, qvec)
        prompt = (f"Question about a loan agreement: {question}\n\nContract excerpts:\n{ctx}\n\n"
                  "List every fact needed to answer the question fully (numbers, rates, floors, dates, day-count rules, "
                  "thresholds). For each, copy the exact words from the excerpts that state it, or use an empty string "
                  "if the excerpts do not state it. Return JSON: "
                  "{\"facts\": [{\"need\": \"short name\", \"quote\": \"exact words or empty\"}]}")
        try:
            r = chat([{"role": "system", "content": "You check whether evidence is complete. Copy quotes exactly."},
                      {"role": "user", "content": prompt}], json_mode=True, max_tokens=600)
            facts = json.loads(r.choices[0].message.content).get("facts", [])
        except Exception:
            return opened
        flat = squash(ctx)
        missing = [f.get("need", "") for f in facts if not f.get("quote") or squash(f["quote"]) not in flat]
        if not missing:
            trail.append("Evidence check: every fact the question needs was found in the opened clauses.")
            return opened
        added = []
        for need in missing[:5]:
            for n in find_seeds(need, 2):
                if n not in opened and n not in added:
                    added.append(n)
        if not added:
            trail.append(f"Evidence check: could not find {', '.join(missing)} anywhere in the graph.")
            return opened
        opened = opened + added[:6]
        trail.append(f"Evidence check: missing {', '.join(missing)}. Opened {', '.join(G.nodes[n]['label'] for n in added[:6])}.")
    return opened


NUM_RE = re.compile(r"\d[\d,]*(?:\.\d+)?")


def numbers_in(text):
    out = set()
    for m in NUM_RE.finditer(text):
        try:
            out.add(float(m.group(0).rstrip(",").replace(",", "")))
        except ValueError:
            pass
    return out


def unexplained_numbers(answer, source_text, calls):
    """Numbers in the answer that appear in no excerpt, no question and no calculator result."""
    allowed = numbers_in(source_text)
    for c in calls:
        for v in list(c["input"].values()) + list(c["output"].values()):
            if isinstance(v, (int, float)):
                allowed.add(float(v))
    bad = []
    for m in NUM_RE.finditer(answer):
        s = m.group(0).rstrip(",").replace(",", "")
        try:
            x = float(s)
        except ValueError:
            continue
        if "." not in s and (x <= 31 or 1900 <= x <= 2100):
            continue                                         # list numbers, small counts, years
        d = len(s.split(".")[1]) if "." in s else 0
        if not any(abs(round(a, d) - x) < 1e-9 for a in allowed):
            bad.append(m.group(0))
    return list(dict.fromkeys(bad))


def ungrounded_inputs(calls, source_text):
    """Contract-derived calculator inputs (spread, day count) that the excerpts never state."""
    allowed, bad = numbers_in(source_text), []
    for c in calls:
        if c["tool"] != "interest_calculator":
            continue
        for key in ("spread_pct", "day_count_basis"):
            v = c["input"].get(key)
            if v is not None and float(v) not in allowed:
                bad.append(f"{key}={v}")
    return bad


def answer_with_checks(system, user, max_fix=1):
    """Answer, then check every number against the excerpts and calculator; ask for a fix if one is unsupported."""
    answer, calls = llm_with_tools(system, user)
    checks = []
    for _ in range(max_fix):
        bad = unexplained_numbers(answer, user, calls)
        weak = ungrounded_inputs(calls, user)
        if not bad and not weak:
            break
        problems = []
        if bad:
            problems.append(f"numbers not found in the excerpts, the question or a calculator result: {', '.join(bad)}")
        if weak:
            problems.append(f"calculator inputs the excerpts do not state: {', '.join(weak)}")
        checks.append("Number check: found " + "; ".join(problems) + ". Asked the model to correct its answer.")
        answer, calls2 = llm_with_tools(
            system, user + "\n\nYour previous answer had these problems: " + "; ".join(problems) +
            ". Write a corrected answer. Use only values stated in the excerpts or the user's assumptions, use the "
            "calculator for computed values, and say plainly if a needed value is missing from the excerpts.")
        calls = calls2
    bad = unexplained_numbers(answer, user, calls)
    if bad:
        answer += f"\n\n_Number check: could not confirm these numbers from the contract text: {', '.join(bad)}._"
        checks.append(f"Number check: still could not confirm {', '.join(bad)}.")
    elif not checks:
        checks.append("Number check: every number in the answer traces to the excerpts or the calculator.")
    return answer, calls, checks


def ask_rag(question, assumptions, k=4):
    hits = retrieve_plain(question, k)
    ctx = "\n\n".join(f"[{c['label']}]\n{c['text']}" for c, _ in hits)
    answer, calls, checks = answer_with_checks(
        SYSTEM_RAG, f"Question: {question}\n\n{assumptions}\n\nContract excerpts:\n{ctx}")
    saw = "\n\n".join(f"--- {c['label']} (similarity {s:.2f}) ---\n{c['text']}" for c, s in hits)
    return dict(answer=answer, calls=calls, saw=saw, checks=checks, retrieved=[c["label"] for c, _ in hits])


def ask_graph(question, assumptions):
    opened, trail = graph_retrieve(question)
    qvec = embed([question])[0]
    opened = complete_evidence(question, opened, qvec, trail)
    parts, ctx = build_context(opened, qvec)
    answer, calls, checks = answer_with_checks(
        SYSTEM_RAG, f"Question: {question}\n\n{assumptions}\n\nContract excerpts:\n{ctx}")
    saw = "\n\n".join(f"--- {label} ---\n{text}" for label, text in parts)
    return dict(answer=answer, calls=calls, saw=saw, trail=trail, checks=checks, opened=opened)

## Step 11. Try the free steps first (no OpenAI key needed)

**What the code below does:** runs Steps 3, 4 and 6 on your PDF *without* calling OpenAI, so you can see the raw material before any AI is involved. In Colab it asks you to upload the contract. It prints how many pages and pieces there are, shows what the term graph says about "Interest Rate", and draws the graph.

**Why we need it:** it lets you check that the contract was read correctly and see the idea of "terms pointing at terms" with your own eyes. You can skip this cell and go straight to the app.

**What to look for:** under "What Interest Rate points to", the list should include Spread, Benchmark and Default Rate. Those are the first links in the chain the lab will follow. The picture is interactive: search "Interest Rate" and click around.

In [ ]:
PDF_PATH = "Sample Loan Agreement 1.pdf"     # change this if your file has another name

try:                                          # in Google Colab, offer an upload box
    from google.colab import files
    if not Path(PDF_PATH).exists():
        uploaded = files.upload()
        PDF_PATH = next(iter(uploaded))
except ImportError:
    pass

if Path(PDF_PATH).exists():
    _pages = read_pdf(PDF_PATH)
    _text = strip_page_furniture(_pages)
    _chunks = make_chunks(_text)
    _G = build_rule_graph(_chunks)
    print(f"{len(_pages)} pages, {len(_text.split()):,} words, cut into {len(_chunks)} pieces")
    print(graph_stats(_G).replace("**", ""))
    term = "term:interest rate"
    if term in _G:
        print("\nWhat Interest Rate points to:", ", ".join(_G.nodes[m]["label"] for m in _G.successors(term)))
        print("Sections that use Interest Rate:", ", ".join(
            _G.nodes[m]["label"] for m in _G.predecessors(term) if _G.nodes[m]["kind"] == "section")[:400], "...")
    from IPython.display import HTML, display
    display(HTML(as_iframe(render_graph_html(_G), 680)))
else:
    print(f"Put the contract PDF next to this notebook as '{PDF_PATH}' (or set PDF_PATH) and run this cell again.")

## Step 12. The Gradio app: the screen people will use

**What the code below does:** builds the web screen. It has four tabs that follow the lesson in order:

1. **Setup**: paste your OpenAI key (hidden as you type), see the model fixed to `gpt-4o-mini`, upload the contract PDF, press **Build**. A live log shows each stage: read the PDF, cut it into pieces, build the search index, run Graphify, merge the graph. Graphify is the slow part, so **Fast demo** is ticked by default.
2. **Knowledge graph**: the interactive graph. Drag, zoom, search, click.
3. **The problem and the fix**: one button that tells the whole story on your contract. It asks "What is the interest rate?", shows a checklist of what each mode got right (formula, Spread value, Benchmark, Index Floor, final rate), and then shows the steps the graph followed to get there.
4. **Ask and compare**: type one question (or click a sample) and see the three answers side by side.

`build_lab()` is the function behind the Build button: it just runs the earlier steps one after another. `compare()` is the function behind the Ask button: it runs `ask_plain`, `ask_rag` and `ask_graph` and formats the results.

**Sample questions** are grouped so you know what each one tests. The first group, **The problem statement**, holds the two prompts that failed in the real contract-review tool. The second group, **Graph showcase**, holds four questions whose answers are spread across several clauses (for example, the lowest possible interest rate needs the Interest Rate, Spread, Benchmark, Term SOFR and Index Floor definitions at once). Then come simple lookups (every mode should cope), other linked-clause questions, calculations, and questions the contract cannot answer. Click one to fill the question box.

Two extras make the comparison fair and measurable: a slider sets how many paragraphs plain RAG may retrieve (raise it and see whether B catches up), and an **automatic test suite** runs test cases through all three modes (a quick set of eight, or all 25) and scores each answer on the facts it should contain.

**What to look for:** after pressing Ask, compare the three columns, then open "what the model saw" under B and C. Under C, read the **trail**: it is the path the graph walk followed.

In [ ]:
import gradio as gr
import pandas as pd

SAMPLES = {
    "The problem statement: the two prompts that failed in the review tool": [
        ("What is the spread and what is the initial benchmark and how are they related? What is the index floor percentage in this agreement?", 4.0),
        ("What is the interest rate in this agreement? Find its definition and calculate the interest rate by finding the values of the terms mentioned in the definition.", 4.0),
    ],
    "Graph showcase: answers are spread across several clauses (try these first)": [
        ("What is the lowest annual interest rate the Borrower could ever pay on this loan while it is performing, "
         "and how much interest is that for a 30-day month on the full loan amount?", 4.0),
        ("What is the Default Rate on this loan if Term SOFR is 0.50%, and how much interest accrues in 30 days "
         "of default on the full loan amount?", 0.5),
        ("Which dates does the very first interest period cover, and when is the first interest payment due after "
         "closing?", 4.0),
        ("To extend the loan a second time, what minimum DSCR and debt yield must the Borrower meet on the First "
         "Extended Maturity Date, and what maximum loan-to-value ratio?", 4.0),
    ],
    "Simple lookups (every mode should manage)": [
        ("Who are the Borrower and the Lender, and what is the maximum loan amount?", 4.0),
        ("What is the Stated Maturity Date?", 4.0),
        ("On which day of the month are payments due, and what if that day is not a Business Day?", 4.0),
        ("What is the Prepayment Premium?", 4.0),
    ],
    "Needs several linked clauses": [
        ("What is the interest rate on this loan and what is it made of?", 4.0),
        ("What happens to the interest rate if Term SOFR falls below the floor?", 4.0),
        ("What is the Default Rate and how is it calculated?", 4.0),
        ("If Term SOFR is discontinued, what rate replaces it?", 4.0),
        ("How is the daily interest computed and what day-count basis applies?", 4.0),
        ("Does this contract use simple interest? Explain how interest is actually calculated.", 4.0),
    ],
    "Calculations (rules from the contract, sums by the calculator)": [
        ("What is the simple interest for a 30-day month on the full loan amount?", 4.0),
        ("Compare the interest for a 30-day month if Term SOFR is 0.50% versus 4.00%.", 4.0),
        ("What would the monthly interest be if the loan were in default and Term SOFR is 4.00%?", 4.0),
    ],
    "Not in the contract (a good answer says so)": [
        ("Who is the CEO of the Lender?", 4.0),
        ("What is the Borrower's credit score?", 4.0),
    ],
}

# Automatic test suite for the SAMPLE contract: each answer is scored on whether it contains the expected facts.
NOT_IN_CONTRACT = r"(not (stated|specified|mentioned|provided|included|contained|found|available)|does not (say|state|specify|contain|identify|provide|mention)|no (information|mention)|cannot (find|determine)|can't (find|determine)|unable to)"
def T(name, kind, q, expect, sofr=4.0, days=30, quick=False):
    return dict(name=name, kind=kind, q=q, expect=expect, sofr=sofr, days=days, quick=quick)


TEST_CASES = [
    # multi-clause: the answer is spread across several clauses
    T("Interest rate with real values, not just the formula", "multi-clause", SAMPLES[list(SAMPLES)[0]][1][0], [r"4\.20", r"1\.12606", r"8\.2"], quick=True),
    T("Spread, benchmark and index floor together", "multi-clause", SAMPLES[list(SAMPLES)[0]][0][0], [r"4\.20", r"(term )?sofr", r"1\.12606"], quick=True),
    T("Lowest possible rate and 30-day interest", "multi-clause", SAMPLES[list(SAMPLES)[1]][0][0], [r"5\.326", r"61328\.6"], quick=True),
    T("Default Rate with SOFR at 0.50%", "multi-clause", SAMPLES[list(SAMPLES)[1]][1][0], [r"10\.326", r"118902\.8"], sofr=0.5, quick=True),
    T("First interest period and first payment", "multi-clause", SAMPLES[list(SAMPLES)[1]][2][0], [r"June 8, 2022", r"July 11, 2022"], quick=True),
    T("Second extension tests", "multi-clause", SAMPLES[list(SAMPLES)[1]][3][0], [r"1\.40", r"8\.25", r"68"], quick=True),
    T("First extension tests", "multi-clause", "To extend the loan the first time, what minimum DSCR and debt yield must the Borrower meet, and what maximum loan-to-value ratio?", [r"1\.30", r"8\.20", r"72"]),
    T("Extension fees, twice", "multi-clause", "What extension fee is payable for each extension, and what is the total if the Borrower extends twice, on the full loan amount?", [r"0\.25", r"69088\.9"]),
    T("Extension dates and notice", "multi-clause", "To what dates can the Stated Maturity Date be extended, and how much notice must the Borrower give?", [r"June 9, 2026", r"June 9, 2027", r"thirty|30", r"ninety|90"]),
    T("What the interest rate is made of", "multi-clause", "What is the interest rate on this loan made of?", [r"4\.20", r"1\.12606", r"Term SOFR"]),
    T("Replacement for Term SOFR", "multi-clause", "If Term SOFR is discontinued, what rate replaces it, and can the replacement go below the floor?", [r"Compounded SOFR", r"ISDA", r"1\.12606"]),
    # calculations
    T("30-day interest, SOFR 4.00%", "calculation", "What is the interest for a 30-day month on the full loan amount if Term SOFR is 4.00%?", [r"8\.2", r"94421\.6"]),
    T("SOFR 0.50% versus 4.00%", "calculation", "Compare the interest for a 30-day month on the full loan amount if Term SOFR is 0.50% versus 4.00%.", [r"61328\.6", r"94421\.6"]),
    # lookups
    T("Spread", "lookup", "What is the Spread?", [r"4\.20"]),
    T("Index Floor", "lookup", "What is the Index Floor, and what does it do?", [r"1\.12606"]),
    T("Day-count basis", "lookup", "What day-count basis is used for interest, and how is the daily rate defined?", [r"360", r"actual"]),
    T("Default Rate", "lookup", "What is the Default Rate and when does it apply?", [r"lesser", r"five percent|5%|5 percent", r"Event of Default"]),
    T("Late Payment Charge", "lookup", "What is the Late Payment Charge?", [r"lesser", r"five percent|5%|5 percent"]),
    T("Prepayment Premium", "lookup", "What is the Prepayment Premium?", [r"1000000"]),
    T("Parties and loan amount", "lookup", "Who are the Borrower and the Lender, and what is the maximum loan amount?", [r"Tides on Boca Raton South Owner", r"ACM CRE Fund I-L", r"13817798"]),
    T("Closing advance", "lookup", "How much did the Lender advance at closing, and how does that compare with the maximum loan amount?", [r"12274640", r"13817798"]),
    T("Maturity date", "lookup", "What is the Stated Maturity Date?", [r"June 9, 2025"], quick=True),
    T("Payment day", "lookup", "On which day of the month are payments due, and what if that day is not a Business Day?", [r"9th|ninth", r"Business Day"]),
    # controls: the contract cannot answer
    T("Not in contract: CEO", "control", "Who is the CEO of the Lender?", [NOT_IN_CONTRACT], quick=True),
    T("Not in contract: credit score", "control", "What is the Borrower's credit score?", [NOT_IN_CONTRACT]),
]


def score_answer(answer, expect):
    text = (answer or "").replace(",", "").replace("$", "")
    return sum(1 for pat in expect if re.search(pat, text, re.I)), len(expect)


def fmt_calls(calls):
    if not calls:
        return "_No calculator was used._"
    lines = []
    for c in calls:
        i, o = c["input"], c["output"]
        if "error" in o:
            lines.append(f"- `{c['tool']}` failed: {o['error']}")
        elif c["tool"] == "interest_calculator":
            lines.append(f"- **interest_calculator**: principal ${i.get('principal', 0):,.0f}, spread {i.get('spread_pct')}%, "
                         f"market rate {i.get('benchmark_pct')}%, floor {i.get('index_floor_pct', 'none')}%, "
                         f"{i.get('days')} days, {i.get('day_count_basis')}-day year. "
                         f"Annual rate **{o.get('annual_rate_pct')}%**, interest **${o.get('interest_for_period', 0):,.2f}**")
        else:
            lines.append(f"- **calculate**: `{i.get('expression')}` = **{o.get('result')}**")
    return "\n".join(lines)


def build_lab(api_key, pdf_file, use_graphify, fast_mode):
    """Runs Steps 3 to 8 one after another and reports progress."""
    log = []
    skip = gr.update()

    def out(msg=None, status=None, stats=None, graph=None):
        if msg:
            log.append(msg)
        return ("\n".join(log[-80:]), skip if status is None else status, skip if stats is None else stats,
                skip if graph is None else graph)

    try:
        if not (api_key or "").strip():
            yield out("Please paste your OpenAI API key first.", status="Waiting for an OpenAI key.")
            return
        if pdf_file is None:
            yield out("Please upload the contract PDF first.", status="Waiting for a PDF.")
            return
        path = getattr(pdf_file, "name", pdf_file)
        LAB.reset()
        LAB.api_key = api_key.strip()
        LAB.client = make_client(LAB.api_key)

        yield out("1/7 Checking the key ...", status="Building ...")
        try:
            embed(["test"])
        except Exception as e:
            yield out(f"The OpenAI key did not work: {e}", status="Key problem. Fix the key and press Build again.")
            return

        pages = read_pdf(path)
        text = strip_page_furniture(pages)
        yield out(f"2/7 Read {len(pages)} pages, {len(text.split()):,} words.")
        LAB.chunks = make_chunks(text)
        kinds = defaultdict(int)
        for c in LAB.chunks:
            kinds[c["kind"]] += 1
        yield out(f"3/7 Cut into {len(LAB.chunks)} pieces ({kinds['definition']} definitions, "
                  f"{kinds['section'] + kinds['article']} sections).")
        if not LAB.chunks:
            yield out("No clauses were found. Is this a text PDF (not a scan)?", status="Could not read the contract.")
            return

        LAB.chunk_vecs = build_vector_index(LAB.chunks)
        yield out("4/7 Built the search index for plain RAG.")

        G = build_rule_graph(LAB.chunks)
        yield out(f"5/7 Built the term graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} links.")

        graphify_path = None
        if use_graphify:
            src, n_files = write_graphify_input(LAB.chunks, fast=fast_mode)
            yield out(f"6/7 Running Graphify on {n_files} files with {MODEL}. This can take a few minutes ...")
            for item in run_graphify(src, LAB.api_key):
                if item[0] == "log":
                    yield out("   graphify: " + item[1][:160])
                else:
                    graphify_path = item[1]
                    if graphify_path is None:
                        yield out(f"   Graphify finished without a graph.json (exit code {item[2]}). "
                                  f"Continuing with the term graph only.")
            if graphify_path:
                info = merge_graphify(G, graphify_path)
                yield out(f"   Merged Graphify's graph: {info['graphify_nodes']} concepts and {info['graphify_edges']} "
                          f"relationships found, {info['new_concept_nodes']} new concepts and {info['new_edges']} new links added.")
        else:
            yield out("6/7 Skipped Graphify (switched off). Using the term graph only.")

        LAB.graph = G
        LAB.graph_html = as_iframe(render_graph_html(G), 680)
        LAB.built = True
        yield out("7/7 Done. Open the other tabs.", status="Ready. Open **Knowledge graph**, **The problem and the fix** or **Ask and compare**.",
                  stats=graph_stats(G), graph=LAB.graph_html)
    except Exception as e:
        yield out(f"Something went wrong: {type(e).__name__}: {e}", status="Build failed. See the log.")


def compare(question, principal, sofr, days, k_rag=4):
    skip = gr.update()

    def out(**kw):
        keys = ["a", "a_calc", "b", "b_calc", "b_saw", "c", "c_calc", "c_trail", "c_saw", "c_graph"]
        return tuple(kw.get(k, skip) for k in keys)

    if not LAB.built:
        msg = "Build the contract first (Setup tab)."
        yield out(a=msg, b=msg, c=msg)
        return
    if not (question or "").strip():
        yield out(a="Type a question first.", b="", c="")
        return
    try:
        assume = assumptions_text(principal, sofr, days)
        yield out(a="Thinking ...", b="Waiting ...", c="Waiting ...")
        A = ask_plain(question, assume)
        yield out(a=A["answer"], a_calc=fmt_calls(A["calls"]), b="Thinking ...")
        B = ask_rag(question, assume, int(k_rag))
        yield out(b=B["answer"], b_calc=fmt_calls(B["calls"]) + "\n\n" + "\n".join(f"_{c}_" for c in B["checks"]), b_saw=B["saw"], c="Following the links in the graph ...")
        C = ask_graph(question, assume)
        sub = set(C["opened"])
        graph = as_iframe(render_graph_html(LAB.graph, subset=sub, highlight=set(C["opened"][:3]), height=380), 440)
        yield out(c=C["answer"], c_calc=fmt_calls(C["calls"]) + "\n\n" + "\n".join(f"_{c}_" for c in C["checks"]),
                  c_trail="\n".join(f"- {t}" for t in C["trail"]),
                  c_saw=C["saw"], c_graph=graph)
    except Exception as e:
        err = f"Something went wrong: {type(e).__name__}: {e}"
        yield out(a=err, b=err, c=err)


def run_tests(principal, k_rag=4, run_all=False):
    """Run the test cases through A, B and C and show a scorecard."""
    if not LAB.built:
        yield "Build the contract first (Setup tab)."
        return
    cases = TEST_CASES if run_all else [t for t in TEST_CASES if t["quick"]]
    rows = []
    passed = {m: defaultdict(int) for m in "ABC"}
    totals = defaultdict(int)
    header = "| Test | Type | A: plain GPT | B: plain RAG | C: graph RAG |\n| --- | --- | --- | --- | --- |\n"
    try:
        for t in cases:
            assume = assumptions_text(principal, t["sofr"], t["days"])
            res = {"A": ask_plain(t["q"], assume), "B": ask_rag(t["q"], assume, int(k_rag)), "C": ask_graph(t["q"], assume)}
            totals[t["kind"]] += 1
            cells = []
            for m in "ABC":
                got, total = score_answer(res[m]["answer"], t["expect"])
                ok = got == total
                passed[m][t["kind"]] += int(ok)
                cells.append(("PASS " if ok else "FAIL ") + f"({got}/{total})")
            rows.append(f"| {t['name']} | {t['kind']} | {cells[0]} | {cells[1]} | {cells[2]} |")
            yield header + "\n".join(rows) + f"\n\n_Ran {len(rows)} of {len(cases)} ..._"
        summary = "\n".join(
            f"| {kind} | {n} | {passed['A'][kind]} | {passed['B'][kind]} | {passed['C'][kind]} |" for kind, n in totals.items())
        yield (header + "\n".join(rows) + "\n\n**Passed by type**\n\n| Type | Cases | A | B | C |\n| --- | --- | --- | --- | --- |\n"
               + summary + "\n\nA test passes when the answer contains every expected fact (for this sample contract). "
               "This is a crude text check, so read the answers too. Results vary a little from run to run.")
    except Exception as e:
        yield f"Something went wrong: {type(e).__name__}: {e}"


SHOWCASE_Q = ("What is the interest rate in this agreement? Find its definition and calculate the interest rate "
              "by finding the values of the terms mentioned in the definition.")

# Generic checklist for the "interest rate" story. A crude text check: it looks for the pieces a lawyer needs.
RATE_CHECKS = [
    ("Gives the formula (Spread plus a benchmark)", r"(sum|total|combination) of[^.\n]{0,40}spread|spread.{0,100}(plus|\+).{0,100}(sofr|benchmark|libor|index)|(sofr|benchmark|libor|index).{0,100}(plus|\+).{0,60}spread"),
    ("States the Spread as a number", r"spread[^.\n]{0,80}?\d+(\.\d+)?\s*%"),
    ("Names the benchmark (for example Term SOFR)", r"term sofr|sofr|libor"),
    ("States the Index Floor as a number", r"floor[^.\n]{0,100}?\d+(\.\d+)?\s*%"),
    ("Gives a final rate as a number", r"(interest rate|annual rate|all-in rate|total rate|rate)[^\n]{0,120}?(is|of|=|:|equals|would be)[^\n]{0,40}?\d+\.\d+\s*%"),
]


def rate_checklist(answers):
    """answers: dict like {"A": text, "B": text, "C": text}. Returns a markdown table of ticks."""
    head = "| What a lawyer needs | A: plain GPT | B: plain RAG | C: graph RAG |\n| --- | --- | --- | --- |\n"
    rows = []
    score = {m: 0 for m in answers}
    for label, pat in RATE_CHECKS:
        cells = []
        for m in "ABC":
            ok = bool(re.search(pat, (answers.get(m) or "").replace(",", ""), re.I | re.S))
            score[m] += int(ok)
            cells.append("YES" if ok else "missing")
        rows.append(f"| {label} | {cells[0]} | {cells[1]} | {cells[2]} |")
    total = len(RATE_CHECKS)
    rows.append(f"| **Score** | **{score['A']} of {total}** | **{score['B']} of {total}** | **{score['C']} of {total}** |")
    return head + "\n".join(rows)


def showcase(question, sofr, k_rag=4):
    """The story: same question, three modes, a checklist, and the graph steps."""
    skip = gr.update()

    def out(**kw):
        keys = ["checklist", "a", "b", "c", "steps", "graph"]
        return tuple(kw.get(k, skip) for k in keys)

    if not LAB.built:
        msg = "Build the contract first (Setup tab)."
        yield out(checklist=msg)
        return
    try:
        q = (question or SHOWCASE_Q).strip()
        assume = assumptions_text(13817798, sofr, 30)
        yield out(checklist="Running mode A (plain GPT) ...", a="Thinking ...", b="Waiting ...", c="Waiting ...", steps="")
        A = ask_plain(q, assume)
        yield out(checklist="Running mode B (plain RAG) ...", a=A["answer"], b="Thinking ...")
        B = ask_rag(q, assume, int(k_rag))
        yield out(checklist="Running mode C (graph RAG) ...", b=B["answer"], c="Following the links in the graph ...")
        C = ask_graph(q, assume)
        steps = "\n".join(f"{i}. {t}" for i, t in enumerate(C["trail"], 1))
        graph = as_iframe(render_graph_html(LAB.graph, subset=set(C["opened"]), highlight=set(C["opened"][:3]), height=380), 440)
        yield out(checklist=rate_checklist({"A": A["answer"], "B": B["answer"], "C": C["answer"]}),
                  c=C["answer"], steps=steps + "\n\n" + fmt_calls(C["calls"]), graph=graph)
    except Exception as e:
        err = f"Something went wrong: {type(e).__name__}: {e}"
        yield out(checklist=err)


def build_ui():
    with gr.Blocks(title="Knowledge Graph Lab: Loan Agreement") as demo:
        gr.Markdown("# Knowledge Graph Lab\nUpload a contract, turn it into a graph, then ask the same question three ways.")
        with gr.Tab("1. Setup"):
            gr.Markdown("Your key is kept in this session's memory only. It is not saved to the notebook.")
            api_key = gr.Textbox(label="OpenAI API key", type="password", placeholder="sk-...")
            gr.Dropdown(choices=[MODEL], value=MODEL, label="Model (fixed for this lab)", interactive=False)
            pdf = gr.File(label="Contract (PDF with selectable text)", file_types=[".pdf"])
            use_gfy = gr.Checkbox(value=True, label="Run Graphify (AI extraction of concepts and links)")
            fast = gr.Checkbox(value=True, label="Fast demo (recommended): Graphify reads only Article 2, the money terms. Untick for the whole contract (slow).")
            build_btn = gr.Button("Build", variant="primary")
            status = gr.Markdown("Not built yet.")
            log = gr.Textbox(label="Progress", lines=12, max_lines=12, interactive=False)

        with gr.Tab("2. Knowledge graph"):
            stats = gr.Markdown("Build the contract first.")
            gr.Markdown("Drag dots, zoom with the wheel, search a term, click a dot to read its clause. "
                        "Try **Interest Rate** and follow the lines.")
            graph_view = gr.HTML()

        with gr.Tab("3. The problem and the fix"):
            gr.Markdown("""### The problem
A law firm uses an AI tool to turn a 200-page loan agreement into a 7-page abstract. It gets the **names, dates and loan amount** right, because each one is written in one place. It gets the **interest rate** wrong, because the rate is not written as a number. It is a recipe spread over several clauses: *Interest Rate = Spread + Benchmark*, and the Benchmark (Term SOFR) can never count as lower than the *Index Floor*. A search that grabs only the "most similar" paragraphs finds the recipe but not the ingredients, so the tool answers with the formula and no values.

### The fix
Turn the contract into a knowledge graph of linked terms. Start at *Interest Rate* and follow the links to *Spread*, *Benchmark*, *Term SOFR* and *Index Floor*, then hand the numbers to a calculator. Press the button to watch it happen on your contract.""")
            sc_q = gr.Textbox(label="Question (this is the prompt that failed in the review tool; you can edit it)", value=SHOWCASE_Q, lines=2)
            with gr.Row():
                sc_sofr = gr.Number(value=4.0, label="Term SOFR for calculations (%)")
                sc_k = gr.Slider(2, 12, value=4, step=1, label="Plain RAG: paragraphs retrieved (k)")
            sc_btn = gr.Button("Run the showcase", variant="primary")
            gr.Markdown("### 1. Scorecard: what each mode gave the lawyer")
            sc_check = gr.Markdown()
            with gr.Row():
                with gr.Column():
                    gr.Markdown("#### A. Plain GPT\n_Has not seen the contract._")
                    sc_a = gr.Markdown()
                with gr.Column():
                    gr.Markdown("#### B. Plain RAG\n_Most similar paragraphs only._")
                    sc_b = gr.Markdown()
                with gr.Column():
                    gr.Markdown("#### C. Graph RAG\n_Linked clauses and a calculator._")
                    sc_c = gr.Markdown()
            gr.Markdown("### 2. How the graph got there (the steps, in order)")
            sc_steps = gr.Markdown()
            gr.Markdown("### 3. The part of the graph that was used (red outline = starting dots)")
            sc_graph = gr.HTML()

        with gr.Tab("4. Ask and compare"):
            gr.Markdown("Ask the **same question** three ways. A has no contract, B gets the most *similar* paragraphs, "
                        "C follows the *links* between clauses and uses a calculator. Open each \"what the model saw\" box.")
            question = gr.Textbox(label="Your question", lines=2,
                                  placeholder="e.g. What is the simple interest for a 30-day month on the full loan amount?")
            with gr.Row():
                principal = gr.Number(value=13817798, label="Principal for calculations ($)")
                sofr = gr.Number(value=4.0, label="Term SOFR for calculations (%)")
                days = gr.Number(value=30, label="Days in the period", precision=0)
                k_rag = gr.Slider(2, 12, value=4, step=1, label="Plain RAG: paragraphs retrieved (k)")
            ask_btn = gr.Button("Ask all three", variant="primary")
            with gr.Row():
                with gr.Column():
                    gr.Markdown("### A. Plain GPT\n_Only the question._")
                    a = gr.Markdown(); a_calc = gr.Markdown()
                with gr.Column():
                    gr.Markdown("### B. Plain RAG\n_The 4 most similar paragraphs._")
                    b = gr.Markdown(); b_calc = gr.Markdown()
                    with gr.Accordion("What the model saw", open=False):
                        b_saw = gr.Textbox(lines=10, max_lines=14, interactive=False, show_label=False)
                with gr.Column():
                    gr.Markdown("### C. Graph RAG\n_Linked clauses + calculator._")
                    c = gr.Markdown(); c_calc = gr.Markdown()
                    c_trail = gr.Markdown()
                    with gr.Accordion("What the model saw", open=False):
                        c_saw = gr.Textbox(lines=10, max_lines=14, interactive=False, show_label=False)
            gr.Markdown("#### The part of the graph used by mode C (red outline = starting dots)")
            c_graph = gr.HTML()
            gr.Markdown("### Sample questions (click one, then press Ask)")
            for title, qs in SAMPLES.items():
                gr.Examples(examples=[[q, s] for q, s in qs], inputs=[question, sofr], label=title)
            with gr.Accordion("Run the automatic test suite (sample contract)", open=False):
                gr.Markdown("Runs test cases through all three modes and scores each answer on the facts it should contain. "
                            "Types: multi-clause (where the graph should help), calculation, lookup, and controls the contract cannot answer.")
                run_all = gr.Checkbox(value=False, label="Run all 25 cases (slower). Unticked runs a quick set of 8.")
                test_btn = gr.Button("Run tests")
                test_out = gr.Markdown()

        sc_btn.click(showcase, [sc_q, sc_sofr, sc_k], [sc_check, sc_a, sc_b, sc_c, sc_steps, sc_graph])
        build_btn.click(build_lab, [api_key, pdf, use_gfy, fast], [log, status, stats, graph_view])
        test_btn.click(run_tests, [principal, k_rag, run_all], test_out)
        ask_btn.click(compare, [question, principal, sofr, days, k_rag],
                      [a, a_calc, b, b_calc, b_saw, c, c_calc, c_trail, c_saw, c_graph])
    return demo

## Step 13. Start the app

**What the code below does:** creates the screen and starts it. `share=True` gives you a public link you can open on any device. You will see the link printed under the cell.

**Things to know:**

- Anyone with the link can open your app, and the app uses whatever key is pasted into it. Do not share the link while your key is in use, and stop the cell when you are done.
- Building the graph with Graphify takes a few minutes on the full contract. For a quicker first run, tick **Fast demo** on the Setup tab.

**What to do next:** open the link, go to **Setup**, paste your key, upload the PDF and press **Build**. Then try the sample questions on the last tab, starting with *"What is the simple interest for a 30-day month on the full loan amount?"*, and compare the three columns.

In [ ]:
demo = build_ui()
demo.queue().launch(share=True, debug=True)

## Answer key for the sample contract (for the person running the demo)

These are the facts in the **sample loan agreement**, so you can judge the three columns. They apply only to that contract. The calculator numbers use the lab's default assumptions: principal $13,817,798 (the *maximum* loan amount; the first advance at closing was smaller, $12,274,640, so real interest depends on the balance), a 30-day month, a 360-day year.

| Question | What a correct answer contains |
| --- | --- |
| Lowest possible rate, 30-day interest | Spread 4.20% plus the Index Floor 1.12606% (Term SOFR can never count as lower): 5.32606%. Interest on $13,817,798 for 30 days on a 360-day year: **$61,328.68**. |
| Default Rate, SOFR 0.50%, 30 days | Interest Rate (floor applies: 5.32606%) plus 5% = 10.32606%, assuming the legal maximum is not lower. Interest: **$118,902.84**. |
| First interest period and first payment | The Initial Payment Date is June 9, 2022 (the first Payment Date after the June 2, 2022 Closing Date), so the first Interest Period runs June 2 to June 8, 2022 and its interest is paid at closing (Section 2.2.1). The first Payment Date after that is July 11, 2022. |
| Second extension tests | On the First Extended Maturity Date: DSCR (Extension Test) at least 1.40:1.00, Debt Yield at least 8.25%, Loan to Value Ratio no more than 68% (Section 2.9). |
| First extension tests | On the original Stated Maturity Date: DSCR (Extension Test) at least 1.30:1.00, Debt Yield at least 8.20%, Loan to Value Ratio no more than 72% (Section 2.9). |
| Extension fees | 0.25% of the maximum Principal for each extension, so 0.50% in total: **$69,088.99** on $13,817,798. |
| Extension dates and notice | To June 9, 2026, then June 9, 2027; notice at least 30 and not more than 90 days before the then-applicable Stated Maturity Date. |
| Late Payment Charge | The lesser of 5% of the unpaid sum or the maximum amount the law permits (Section 2.2.5). |
| Prepayment Premium | The positive difference, if any, of $1,000,000 less the regularly scheduled interest payments made so far (with listed exclusions). |
| Closing advance | $12,274,640 at closing, against a maximum principal of $13,817,798. |
| Parties and maximum loan amount | Borrower: Tides on Boca Raton South Owner, LLC. Lender: ACM CRE Fund I-L, LP. Maximum principal $13,817,798. |
| Stated Maturity Date | June 9, 2025, as it may be extended under Section 2.9. |
| Payment day | The 9th of each month, or the next Business Day if the 9th is not one. |
| What the interest rate is made of | Spread (4.20%) plus the Benchmark. The Benchmark is initially one-month Term SOFR, which can never count as lower than the Index Floor (1.12606%). |
| SOFR below the floor | The floor applies: the rate is 4.20% + 1.12606% = 5.32606%, however low SOFR falls. |
| Day count | Actual days in the month over a 360-day year; the daily rate is 1/360 of the Interest Rate; interest is based on the average daily principal balance (Section 2.2.3). |
| Is it simple interest? | The contract never uses that phrase. Interest is a floating rate accruing daily on an actual/360 basis. |
| 30-day interest, SOFR 4.00% | 8.20% annual rate, interest **$94,421.62**. |
| 30-day interest, SOFR 0.50% | Floor applies: 5.32606% annual rate, interest **$61,328.68** (versus $94,421.62 at 4.00%). |
| 30-day interest in default, SOFR 4.00% | Default Rate is the lesser of the legal maximum and the Interest Rate plus 5%, so 13.20%, interest **$151,995.78** (assuming the legal maximum is not lower). |
| If Term SOFR is discontinued | The Benchmark Replacement, in order: Compounded SOFR plus adjustment; the rate selected by the Relevant Governmental Body plus adjustment; the ISDA Fallback Rate plus adjustment; a rate chosen by the Lender. Never below the Index Floor (Section 2.10). |
| CEO of the Lender, Borrower's credit score | Not in the contract. A good answer says so. |

**What to expect:** mode A cannot know the Spread or the floor, so it should refuse or guess. Mode B may find the "Interest Rate" definition but can miss the floor or the day count. Mode C is designed to collect every linked clause. We have not promised any of these outcomes in advance: run the questions and report what you actually see.